# CIFAR-10 Preprocessing for k-Sparse Autoencoder

Implements the preprocessing pipeline from Coates et al. 2011, as used in Makhzani & Frey 2014:
1. Extract 1,000,000 random 8×8 patches from CIFAR-10 training images
2. Apply Local Contrast Normalization (LCN) per patch
3. Apply ZCA Whitening across the full patch set

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from preprocessing import (
    load_cifar10,
    extract_patches,
    local_contrast_normalize,
    compute_zca_params,
    apply_zca_whitening,
    save_whitening_params,
    load_whitening_params,
    preprocess_pipeline,
)
from verify import (
    verify_lcn,
    verify_zca,
    plot_patches,
    plot_eigenspectrum,
    plot_pixel_histograms,
    plot_covariance_matrix,
)

# Config
DATA_DIR    = "./data"
OUTPUT_DIR  = "./outputs"
N_PATCHES   = 1_000_000
PATCH_SIZE  = 8
SEED        = 42

print("Imports OK")

## 1. Load CIFAR-10

In [ ]:
images = load_cifar10(DATA_DIR)
print(f"Images shape: {images.shape}, dtype: {images.dtype}")

# Display 20 sample images
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
rng = np.random.default_rng(0)
sample_idx = rng.integers(0, len(images), 20)
for ax, idx in zip(axes.flat, sample_idx):
    ax.imshow(images[idx])
    ax.axis("off")
plt.suptitle("20 Sample CIFAR-10 Training Images", fontsize=12)
plt.tight_layout()
plt.show()

## 2. Extract Random Patches

In [ ]:
patches_raw = extract_patches(images, n_patches=N_PATCHES, patch_size=PATCH_SIZE, seed=SEED)
print(f"Patches shape: {patches_raw.shape}, dtype: {patches_raw.dtype}")
print(f"Value range: [{patches_raw.min():.1f}, {patches_raw.max():.1f}]")

plot_patches(patches_raw, n=20, patch_size=PATCH_SIZE, title="20 Raw Patches (uint8 values as float)")
plt.show()

## 3. Local Contrast Normalization (LCN)

In [ ]:
patches_lcn = local_contrast_normalize(patches_raw)
print(f"LCN patches shape: {patches_lcn.shape}")

plot_patches(patches_lcn, n=20, patch_size=PATCH_SIZE, title="20 Patches After LCN")
plt.show()

lcn_results = verify_lcn(patches_lcn)
print(lcn_results)

## 4. ZCA Whitening

In [ ]:
params = compute_zca_params(patches_lcn)
print(f"W shape: {params['W'].shape}")
print(f"Eigenvalue range: [{params['s'].min():.4f}, {params['s'].max():.4f}]")

patches_white = apply_zca_whitening(patches_lcn, params)
print(f"Whitened patches shape: {patches_white.shape}")

plot_patches(patches_white, n=20, patch_size=PATCH_SIZE, title="20 Patches After ZCA Whitening")
plt.show()

zca_results = verify_zca(patches_white)
print(zca_results)

## 5. Visualizations

In [ ]:
plot_eigenspectrum(params)
plt.show()

In [ ]:
plot_pixel_histograms(patches_raw, patches_lcn, patches_white)
plt.show()

In [ ]:
plot_covariance_matrix(patches_lcn, patches_white)
plt.show()

## 6. Save Artifacts

In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

save_whitening_params(params, os.path.join(OUTPUT_DIR, "whitening_params.npz"))
np.save(os.path.join(OUTPUT_DIR, "patches_whitened.npy"), patches_white)
print("Artifacts saved.")

## 7. Roundtrip Test

Verify we can recover LCN patches from whitened patches by applying the inverse of W.

In [ ]:
# Reload params from disk to test full roundtrip
params_loaded = load_whitening_params(os.path.join(OUTPUT_DIR, "whitening_params.npz"))

W = params_loaded["W"]          # (192, 192)
mean = params_loaded["mean"]    # (192,)

# Inverse of W: since W = U diag(...) U^T, W^{-1} = U diag(...^{-1}) U^T
# But easier: just use np.linalg.inv(W)
W_inv = np.linalg.inv(W)

n_test = 10_000
X_white = patches_white[:n_test].astype(np.float64)
X_recovered = X_white @ W_inv + mean

X_lcn = patches_lcn[:n_test].astype(np.float64)
max_err = np.abs(X_recovered - X_lcn).max()
allclose = np.allclose(X_recovered, X_lcn, atol=1e-3)

print(f"Max absolute error: {max_err:.2e}")
print(f"np.allclose(recovered, lcn, atol=1e-3): {allclose}")

## 8. Full Pipeline (Alternative: run everything at once)

In [ ]:
# Uncomment to re-run everything from scratch:
# patches_white, params = preprocess_pipeline(
#     data_dir=DATA_DIR,
#     output_dir=OUTPUT_DIR,
#     n_patches=N_PATCHES,
#     patch_size=PATCH_SIZE,
#     seed=SEED,
# )